# Task4 — ConvNeXt architecture diversity

เปลี่ยน backbone เป็น **ConvNeXt-Tiny** เพื่อทดสอบว่าแก้ข้อผิดพลาดที่ ResNet/EfficientNet ร่วมกันทำพลาดได้หรือไม่ ไม่รับรองว่าคะแนนจะเพิ่ม

อ้างอิง: [A ConvNet for the 2020s, CVPR 2022](https://openaccess.thecvf.com/content/CVPR2022/html/Liu_A_ConvNet_for_the_2020s_CVPR_2022_paper.html). งานวิจัยเสนอ large-kernel depthwise convolution และ LayerNorm; ประโยชน์ต่อข้อมูลการแข่งขันนี้ยังเป็นสมมติฐาน

ใช้ torchvision ImageNet pretrained ตามข้อยกเว้น Task4; ไม่มีข้อมูลภายนอกเพิ่มเติม. Train-only 5-fold, 18 epochs/fold, seed42, batch16, weighted CE, compound augmentation .6, LR backbone1e-4/head5e-4, cosine. เปลี่ยน architecture และ batch size จึงไม่ใช่การแยกผล architecture อย่างสมบูรณ์

เปรียบเทียบ **baseline ที่เคยได้ Public 0.81047**, ConvNeXt เดี่ยว และ **baseline 75% + ConvNeXt 25%** (ResNet compound18.75%, sampler18.75%, EfficientNet37.5%, ConvNeXt25%). น้ำหนักกำหนดก่อนรัน

Run All ฝึก 5 folds; full refit อีก 18 epochs เฉพาะผ่าน development/audit. ไม่เปิด Test และไม่สร้าง submission. เป็นงานหนักกว่ารอบ EfficientNet; ใช้เวลา epoch จริงประมาณเวลาที่เหลือ ไม่สัญญาเวลาเทรน

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_convnext_training as training
from Task4.models import image_convnext_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/convnext_diversity_v1'
BATCH_SIZE=16
NUM_WORKERS=0
RUN_FULL_REFIT=True
NAME='convnext_compound18'
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')
display(asdict(training.RECIPES[NAME]))

Device: mps


{'architecture': 'convnext_tiny',
 'epochs': 18,
 'backbone_lr': 0.0001,
 'head_lr': 0.0005,
 'weight_decay': 0.0001,
 'schedule': 'cosine',
 'warmup_epochs': 1,
 'augmentation': 'compound_corruption',
 'label_smoothing': 0.0,
 'seed': 42,
 'balance': 'weighted_loss',
 'jsd_weight': 0.0,
 'compound_probability': 0.6}

## 1. ตรวจ Train และ reference
ตรวจแถว fold, image hashes, group separation และโมเดลเดิมทั้ง 15 ตัวก่อนเริ่ม

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':asdict(training.RECIPES[NAME]),'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. Train และ paired validation
มีเซลล์ฝึกเพียงเซลล์เดียว. เซฟทุก epoch และโหลด fold ที่เสร็จแล้ว. หากหยุดกลาง epoch จะทำ epoch นั้นใหม่. ไม่เปลี่ยน batch/recipe หลังเริ่ม run; ถ้าต้องเปลี่ยนให้ใช้ OUT ใหม่

In [3]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nConvNeXt fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*training.RECIPES[NAME].epochs*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


ConvNeXt fold 1/5
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /Users/meng/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:07<00:00, 15.4MB/s] 


fold1 epoch 1 / 18 loss 0.40099 seconds 706.3
fold1 epoch 2 / 18 loss 0.19354 seconds 603.2


KeyboardInterrupt: 

## 3. Evidence และ selection
คงเกณฑ์เดิม ไม่ลดเกณฑ์หลังเห็นผล: original ≥−.001; family-balanced stress ≥+.002, ชนะ ≥4/5 folds; worst class ≥−.01; worst condition ≥−.02. เกณฑ์เป็น heuristic ไม่ใช่ statistical significance. OOF และ corruption suite ถูกใช้ซ้ำจึงเป็น exploratory

In [ ]:
display(evaluation.summarize(all_scores))
display(all_overlap[all_overlap.condition.eq('original')])
selection=evaluation.select(all_scores)
validation.write_json(OUT/'development_selection.json',selection)
display(selection)

original_overlap=all_overlap[all_overlap.condition.eq('original')]
display(original_overlap)
print('Total candidate fixes:',int(original_overlap.candidate_fixes.sum()))
print('Total candidate breaks:',int(original_overlap.candidate_breaks.sum()))
print('Shared errors:',int(original_overlap.both_wrong.sum()))


## 4. Audit veto
ใช้เฉพาะสูตรที่เลือกจาก development; ไม่เลือกสูตรใหม่ตาม audit. ชุดนี้เคยใช้แล้ว ไม่ใช่ untouched holdout

In [ ]:
audit_result={'passed':False,'reason':'Development did not select a candidate'}
if selection['chosen']!='baseline':
    audit_parts=[]
    for fold in range(1,6):
        score,_=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold],phase='audit')
        audit_parts.append(score)
    audit_scores=pd.concat(audit_parts,ignore_index=True)
    audit_scores.to_parquet(OUT/'audit_scores.parquet',index=False)
    audit_result=evaluation.audit(audit_scores,selection['chosen'])
    display(evaluation.summarize(audit_scores))
validation.write_json(OUT/'audit_decision.json',audit_result)
display(audit_result)

## 5. Full refit และ save model
เมื่อผ่านทั้งสองขั้น ฝึก ConvNeXt บน Train ทั้งหมด. บันทึก model.pt และ readiness สำหรับตรวจผลก่อน inference. ถ้าไม่ผ่านจะคง baseline เดิม ไม่มี submission ใหม่

In [ ]:
full_path=None
if RUN_FULL_REFIT and selection['chosen']!='baseline' and audit_result['passed']:
    full_path=training.fit_fixed(RAW,groups,training.RECIPES[NAME],OUT/NAME/'full',BATCH_SIZE,NUM_WORKERS)
    validation.write_json(OUT/'ready_for_review.json',{
        'procedure':selection['chosen'],'convnext_model':str(full_path),'sha256':validation.sha(full_path),
        'test_used':False,'submission_created':False,
        'selection':selection,'audit':audit_result})
    print('Saved full model:',full_path)
else:
    print('No full refit: inspect development/audit results above. Existing baseline preserved.')